# BirdCLEF+ 2026 — Starter Notebook
**Goal:** Build a working end-to-end submission using Google's Perch v2 model as a frozen feature extractor, with a trainable classification head on top.

**Pipeline overview:**
1. Load train metadata → build a label map
2. Load audio clips → extract 1024-dim Perch embeddings
3. Train a small MLP head on the embeddings
4. At inference: slice test soundscapes into 5-second windows → embed → predict → format submission

**Why Perch?** It was pretrained on a large global bird audio dataset, so its embeddings already capture bird vocalization structure. You don't need to learn audio features from scratch.

---
## Cell 1 — Install dependencies
Perch is loaded via the `perch-hoplite` package. Run this cell once; it takes ~2 minutes.

In [1]:
!pip install -q git+https://github.com/google-research/perch-hoplite.git
!pip install -q librosa soundfile

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This b

---
## Cell 2 — Imports and config
All tunable parameters live in `CFG` so they're easy to find and change.

In [2]:
import os
import glob
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# ── Config ────────────────────────────────────────────────────────────────────
class CFG:
    DATA_DIR      = '/kaggle/input/competitions/birdclef-2026'
    TRAIN_AUDIO   = '/kaggle/input/competitions/birdclef-2026/train_audio'
    TEST_AUDIO    = '/kaggle/input/competitions/birdclef-2026/test_soundscapes'
    TRAIN_CSV     = '/kaggle/input/competitions/birdclef-2026/train.csv'
    SAMPLE_SUB    = '/kaggle/input/competitions/birdclef-2026/sample_submission.csv'

    # Audio
    SAMPLE_RATE   = 32000
    CLIP_DURATION = 5
    CLIP_SAMPLES  = SAMPLE_RATE * CLIP_DURATION

    # Perch embedding
    EMBED_DIM = 1024  
    
    # Training
    EPOCHS        = 20
    BATCH_SIZE    = 256
    LR            = 1e-3
    N_FOLDS       = 5
    TRAIN_FOLD    = 0
    SEED          = 42

    DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
    
print(f"Using device: {CFG.DEVICE}")
torch.manual_seed(CFG.SEED)
np.random.seed(CFG.SEED)

Using device: cpu


---
## Cell 3 — Load metadata and build label map
Each species is identified by a 6-letter eBird code (e.g. `'rufpic1'`). We map these to integer indices.

In [3]:
train_df = pd.read_csv(CFG.TRAIN_CSV)

# primary_label is the numeric taxon ID string — use it as the class identifier
all_species = sorted(train_df['primary_label'].astype(str).unique().tolist())
NUM_CLASSES = len(all_species)
label2idx   = {sp: i for i, sp in enumerate(all_species)}
idx2label   = {i: sp for sp, i in label2idx.items()}

print(f"Number of classes : {NUM_CLASSES}")
print(f"Classes (first 5) : {all_species[:5]}")

# Add integer label index
train_df['primary_label'] = train_df['primary_label'].astype(str)
train_df['label_idx']     = train_df['primary_label'].map(label2idx)

train_df['filepath'] = CFG.TRAIN_AUDIO + '/' + train_df['filename']

Number of classes : 206
Classes (first 5) : ['1161364', '116570', '1176823', '1595929', '209233']


---
## Cell 4 — Load Perch v2 and test it
`perch_hoplite` downloads the model weights from Kaggle Models automatically on first run (~500 MB). After that it loads from cache.

Perch takes a **160,000-sample (5-second @ 32 kHz) mono float32 array** and returns:
- `outputs.embeddings` — shape `(1, 1280)` — the feature vector we'll use
- `outputs.logits['label']` — Perch's own bird-species predictions (useful later)

In [4]:
from perch_hoplite.zoo import model_configs

print("Loading BirdNET v2.3 model...")
birdnet_model = model_configs.load_model_by_name('birdnet_V2.3')
print("BirdNET v2.3 loaded.")

# Smoke test
dummy_audio = np.zeros(CFG.CLIP_SAMPLES, dtype=np.float32)
dummy_output = birdnet_model.embed(dummy_audio)
print(f"Embedding shape : {dummy_output.embeddings.shape}")
print(f"Logits keys     : {list(dummy_output.logits.keys())}")

# Use this model going forward
perch_model = birdnet_model

2026-05-30 23:23:44.590949: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780183424.909984      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780183425.003642      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780183425.806151      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780183425.806278      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780183425.806286      16 computation_placer.cc:177] computation placer alr

Loading BirdNET v2.3 model...
BirdNET v2.3 loaded.
Embedding shape : (1, 1, 1024)
Logits keys     : ['birdnet_v2_1']


---
## Cell 5 — Audio loading helper
This helper loads an `.ogg` file, resamples to 32 kHz if needed, converts to mono, and pads/crops to exactly `CLIP_DURATION` seconds.

**Why pad/crop?**  
Training clips vary in length. Perch needs exactly 160,000 samples. Shorter clips are zero-padded; longer clips are cropped from the centre.

In [5]:
def load_audio_clip(filepath: str, target_sr: int = CFG.SAMPLE_RATE,
                    duration: int = CFG.CLIP_DURATION) -> np.ndarray:
    """
    Load an audio file and return a float32 array of exactly
    (target_sr * duration) samples, mono.
    """
    target_len = target_sr * duration

    # librosa handles .ogg, .mp3, .wav and resamples automatically
    audio, sr = librosa.load(filepath, sr=target_sr, mono=True)

    if len(audio) < target_len:
        # Short clip: zero-pad on the right
        audio = np.pad(audio, (0, target_len - len(audio)))
    else:
        # Long clip: take a centre crop
        start = (len(audio) - target_len) // 2
        audio = audio[start : start + target_len]

    return audio.astype(np.float32)


# Test it on the first training file
test_clip = load_audio_clip(train_df['filepath'].iloc[0])
print(f"Loaded clip shape : {test_clip.shape}  (expected {CFG.CLIP_SAMPLES})")
print(f"Value range       : [{test_clip.min():.3f}, {test_clip.max():.3f}]")

Loaded clip shape : (160000,)  (expected 160000)
Value range       : [-0.157, 0.159]


---
## Cell 6 — Pre-compute Perch embeddings for all training clips
We extract embeddings **once** and save them to disk. This means training the classification head is just matrix operations — no audio I/O during the training loop.

⏱ **Expected time:** ~15–30 min on a Kaggle GPU (Perch runs on CPU for the embedding step; the bottleneck is audio I/O and TensorFlow forward passes).

💡 **If you're short on time:** set `MAX_FILES_PER_CLASS` to a small number (e.g. 50) to embed a subset first, verify the pipeline works, then run on the full dataset.

In [6]:
import pickle

EMBED_CACHE = '/kaggle/working/train_embeddings.pkl'
MAX_FILES_PER_CLASS = 50  # Set to e.g. 50 for a quick test run; None = use all

def extract_embeddings(df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """
    Run Perch over every row in df.
    Returns:
        embeddings : float32 array of shape (N, EMBED_DIM)
        labels     : int array of shape (N,)
    """
    embeddings = []
    labels = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Embedding"):
        try:
            audio = load_audio_clip(row['filepath'])
            output = perch_model.embed(audio)
            emb = np.array(output.embeddings).squeeze()  # shape: (1280,)
            embeddings.append(emb)
            labels.append(row['label_idx'])
        except Exception as e:
            print(f"  Skipping {row['filepath']}: {e}")

    return np.array(embeddings, dtype=np.float32), np.array(labels, dtype=np.int64)


if os.path.exists(EMBED_CACHE):
    print("Loading cached embeddings...")
    with open(EMBED_CACHE, 'rb') as f:
        train_embeddings, train_labels = pickle.load(f)
    print(f"Loaded  embeddings: {train_embeddings.shape}")
else:
    print("Extracting embeddings (this takes a while — go grab a coffee ☕)")

    run_df = train_df.copy()
    if MAX_FILES_PER_CLASS is not None:
        run_df = (
            run_df.groupby('primary_label', group_keys=False)
                  .apply(lambda g: g.sample(min(len(g), MAX_FILES_PER_CLASS),
                                            random_state=CFG.SEED))
                  .reset_index(drop=True)
        )
        print(f"Using subset: {len(run_df)} clips ({MAX_FILES_PER_CLASS} per class max)")

    train_embeddings, train_labels = extract_embeddings(run_df)

    with open(EMBED_CACHE, 'wb') as f:
        pickle.dump((train_embeddings, train_labels), f)
    print(f"\nEmbeddings saved to {EMBED_CACHE}")
    print(f"Shape: {train_embeddings.shape}")

Extracting embeddings (this takes a while — go grab a coffee ☕)
Using subset: 8473 clips (50 per class max)


Embedding:   0%|          | 0/8473 [00:00<?, ?it/s]


Embeddings saved to /kaggle/working/train_embeddings.pkl
Shape: (8473, 1024)


---
## Cell 7 — PyTorch Dataset and MLP head
The classification head is deliberately simple: two linear layers with a ReLU in the middle. This is enough to achieve a solid baseline score. You can make it deeper later if needed.

In [7]:
class EmbeddingDataset(Dataset):
    """Simple dataset that wraps pre-computed embeddings and integer labels."""

    def __init__(self, embeddings: np.ndarray, labels: np.ndarray):
        self.embeddings = torch.from_numpy(embeddings)   # (N, 1280)
        self.labels     = torch.from_numpy(labels)       # (N,)  — integer class index

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]


class MLPHead(nn.Module):
    """
    Two-layer MLP classification head.
      Input  : 1280-dim Perch embedding
      Output : NUM_CLASSES raw logits (no sigmoid — BCEWithLogitsLoss applies it internally)
    """

    def __init__(self, in_dim: int = CFG.EMBED_DIM, hidden_dim: int = 512,
                 num_classes: int = NUM_CLASSES, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)    # shape: (batch, num_classes)


# Quick sanity check
dummy_emb = torch.randn(4, CFG.EMBED_DIM)
model_check = MLPHead()
out = model_check(dummy_emb)
print(f"MLPHead output shape: {out.shape}  (expected [4, {NUM_CLASSES}])")

MLPHead output shape: torch.Size([4, 206])  (expected [4, 206])


---
## Cell 8 — Training loop
We use a single train/val split derived from a stratified k-fold. The loss is `BCEWithLogitsLoss`, which is the right choice for multilabel classification — it applies a sigmoid per class and computes binary cross-entropy independently for each.

After each epoch we report the macro-average ROC-AUC on the validation fold, which is the actual competition metric.

In [8]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR


def make_one_hot(labels: np.ndarray, num_classes: int) -> np.ndarray:
    """Convert integer class indices to one-hot float32 targets."""
    one_hot = np.zeros((len(labels), num_classes), dtype=np.float32)
    one_hot[np.arange(len(labels)), labels] = 1.0
    return one_hot


def compute_macro_auc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """Macro-averaged ROC-AUC, skipping classes with no positive samples."""
    aucs = []
    for c in range(y_true.shape[1]):
        if y_true[:, c].sum() > 0:
            aucs.append(roc_auc_score(y_true[:, c], y_score[:, c]))
    return float(np.mean(aucs)) if aucs else 0.0


# ── Build train / val split ────────────────────────────────────────────────
skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
splits = list(skf.split(train_embeddings, train_labels))
train_idx, val_idx = splits[CFG.TRAIN_FOLD]

X_train, y_train = train_embeddings[train_idx], train_labels[train_idx]
X_val,   y_val   = train_embeddings[val_idx],   train_labels[val_idx]

y_train_oh = make_one_hot(y_train, NUM_CLASSES)
y_val_oh   = make_one_hot(y_val,   NUM_CLASSES)

train_ds = EmbeddingDataset(X_train, y_train)
val_ds   = EmbeddingDataset(X_val,   y_val)
train_dl = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(train_ds):,} samples | Val: {len(val_ds):,} samples")

# ── Initialise model, loss, optimiser ─────────────────────────────────────
model     = MLPHead().to(CFG.DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=CFG.LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS)

best_auc       = 0.0
best_model_path = '/kaggle/working/best_mlp_head.pt'

# ── Training loop ─────────────────────────────────────────────────────────
for epoch in range(1, CFG.EPOCHS + 1):

    # --- Train ---
    model.train()
    train_loss = 0.0
    for embs, lbls in train_dl:
        embs  = embs.to(CFG.DEVICE)
        # Build one-hot targets on the fly for this batch
        targets = torch.zeros(len(lbls), NUM_CLASSES, device=CFG.DEVICE)
        targets.scatter_(1, lbls.unsqueeze(1).to(CFG.DEVICE), 1.0)

        optimizer.zero_grad()
        logits = model(embs)
        loss   = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(lbls)

    train_loss /= len(train_ds)
    scheduler.step()

    # --- Validate ---
    model.eval()
    all_logits = []
    with torch.no_grad():
        for embs, _ in val_dl:
            logits = model(embs.to(CFG.DEVICE))
            all_logits.append(torch.sigmoid(logits).cpu().numpy())

    val_probs = np.concatenate(all_logits, axis=0)   # (N_val, NUM_CLASSES)
    val_auc   = compute_macro_auc(y_val_oh, val_probs)

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), best_model_path)
        flag = ' ← best'
    else:
        flag = ''

    print(f"Epoch {epoch:02d}/{CFG.EPOCHS}  "
          f"train_loss={train_loss:.4f}  "
          f"val_auc={val_auc:.4f}{flag}")

print(f"\nBest validation macro-AUC: {best_auc:.4f}")

Train: 6,778 samples | Val: 1,695 samples
Epoch 01/20  train_loss=0.1603  val_auc=0.5244 ← best
Epoch 02/20  train_loss=0.0462  val_auc=0.6056 ← best
Epoch 03/20  train_loss=0.0299  val_auc=0.7328 ← best
Epoch 04/20  train_loss=0.0239  val_auc=0.8230 ← best
Epoch 05/20  train_loss=0.0195  val_auc=0.8732 ← best
Epoch 06/20  train_loss=0.0165  val_auc=0.8993 ← best
Epoch 07/20  train_loss=0.0143  val_auc=0.9159 ← best
Epoch 08/20  train_loss=0.0129  val_auc=0.9256 ← best
Epoch 09/20  train_loss=0.0117  val_auc=0.9335 ← best
Epoch 10/20  train_loss=0.0108  val_auc=0.9378 ← best
Epoch 11/20  train_loss=0.0102  val_auc=0.9413 ← best
Epoch 12/20  train_loss=0.0096  val_auc=0.9433 ← best
Epoch 13/20  train_loss=0.0091  val_auc=0.9448 ← best
Epoch 14/20  train_loss=0.0088  val_auc=0.9456 ← best
Epoch 15/20  train_loss=0.0085  val_auc=0.9465 ← best
Epoch 16/20  train_loss=0.0083  val_auc=0.9468 ← best
Epoch 17/20  train_loss=0.0082  val_auc=0.9472 ← best
Epoch 18/20  train_loss=0.0081  val_auc=

---
## Cell 9 — Inference on test soundscapes
Test data is long continuous recordings (each could be 10–60 minutes long). We need to:
1. Slice each recording into non-overlapping 5-second windows
2. Embed each window with Perch
3. Run the MLP head to get class probabilities
4. Format the result as `<filename>_<start_time>` rows matching `sample_submission.csv`

**Time limit:** Kaggle runs inference CPU-only with ~90 minutes. Perch + a tiny MLP is well within budget.

In [9]:
def predict_soundscape(filepath: str) -> pd.DataFrame:
    """
    Slice a soundscape into 5-second windows, embed each with Perch,
    run the MLP head, and return a DataFrame with row_id + class probabilities.
    """
    audio, sr = librosa.load(filepath, sr=CFG.SAMPLE_RATE, mono=True)
    stem = os.path.splitext(os.path.basename(filepath))[0]

    rows = []
    start = 0
    while start + CFG.CLIP_SAMPLES <= len(audio):
        clip = audio[start : start + CFG.CLIP_SAMPLES].astype(np.float32)
        output = perch_model.embed(clip)
        emb = torch.from_numpy(np.array(output.embeddings).squeeze()).unsqueeze(0)

        with torch.no_grad():
            probs = torch.sigmoid(model(emb.to(CFG.DEVICE))).cpu().numpy()[0]

        row = {'row_id': f"{stem}_{start // CFG.SAMPLE_RATE}"}
        row.update({idx2label[i]: float(probs[i]) for i in range(NUM_CLASSES)})
        rows.append(row)
        start += CFG.CLIP_SAMPLES

    return pd.DataFrame(rows)

In [10]:
# Load the best checkpoint
model.load_state_dict(torch.load(best_model_path, map_location=CFG.DEVICE))
model.eval()

# Inspect the sample submission to know the expected row_id format
sample_sub = pd.read_csv(CFG.SAMPLE_SUB)
print("Sample submission shape:", sample_sub.shape)
print(sample_sub.head(3))

# The row_id format is typically: <soundscape_stem>_<start_second>
# e.g.  soundscape_29197_5  means file soundscape_29197.ogg, window starting at 5s

Sample submission shape: (3, 235)
                                    row_id   1161364    116570   1176823  \
0   BC2026_Test_0001_S05_20250227_010002_5  0.004274  0.004274  0.004274   
1  BC2026_Test_0001_S05_20250227_010002_10  0.004274  0.004274  0.004274   
2  BC2026_Test_0001_S05_20250227_010002_15  0.004274  0.004274  0.004274   

    1491113   1595929    209233     22930     22956     22961  ...   whnjay1  \
0  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  ...  0.004274   
1  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  ...  0.004274   
2  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  ...  0.004274   

     whtdov   whwpic1    y00678    yebcar   yebela1    yecmac    yecpar  \
0  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274   
1  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274   
2  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274   

    yehcar1   yeofly1  
0  0.004274  0.

In [11]:
# During development, test_soundscapes only has a readme.txt
# Real test audio is injected by Kaggle at submission time only.
# We validate the inference pipeline using training files as stand-ins.

test_files = sorted(glob.glob(os.path.join(CFG.TEST_AUDIO, '*.ogg')))

if len(test_files) == 0:
    print("No test soundscapes found (expected during development).")
    print("Running inference on 2 training files as a pipeline check...\n")
    test_files = train_df['filepath'].head(2).tolist()
    is_dev_mode = True
else:
    is_dev_mode = False

all_preds = []
for fp in tqdm(test_files, desc="Soundscapes"):
    try:
        df_pred = predict_soundscape(fp)
        all_preds.append(df_pred)
    except Exception as e:
        print(f"  Error on {fp}: {e}")

if all_preds:
    submission = pd.concat(all_preds, ignore_index=True)
    print(f"Prediction rows : {len(submission):,}")
    print(submission.head(3))
    if is_dev_mode:
        print("\nPipeline check passed. Row IDs will use training filenames as stems.")
        print("At submission time these will be real soundscape IDs matching sample_submission.csv")
else:
    print("No predictions generated — check errors above.")

No test soundscapes found (expected during development).
Running inference on 2 training files as a pipeline check...



Soundscapes:   0%|          | 0/2 [00:00<?, ?it/s]

Prediction rows : 8
           row_id   1161364    116570   1176823   1595929    209233     22930  \
0   iNat1216197_0  0.008408  0.000004  0.000028  0.000106  0.000005  0.000043   
1   iNat1216197_5  0.005908  0.000004  0.000116  0.000148  0.000005  0.000071   
2  iNat1216197_10  0.008172  0.000006  0.000180  0.000121  0.000003  0.000069   

      22956     22961     22967  ...   whnjay1    whtdov   whwpic1    y00678  \
0  0.000044  0.000054  0.000038  ...  0.000015  0.000467  0.000620  0.000042   
1  0.000561  0.000184  0.000097  ...  0.000023  0.000258  0.000194  0.000184   
2  0.000111  0.000106  0.000306  ...  0.000036  0.000484  0.000463  0.000128   

     yebcar   yebela1    yecmac    yecpar   yehcar1   yeofly1  
0  0.000028  0.000049  0.000004  0.000012  0.000344  0.000148  
1  0.000074  0.000067  0.000015  0.000040  0.000790  0.000013  
2  0.000148  0.000225  0.000019  0.000060  0.000529  0.000042  

[3 rows x 207 columns]

Pipeline check passed. Row IDs will use training file

---
## Cell 10 — Align with sample_submission and save
The sample submission defines exactly which `row_id` values are expected and in what column order. We reindex to match it exactly — missing rows get 0.0, extra rows are dropped.

In [12]:
# Align columns and rows to the sample submission exactly
sample_sub = pd.read_csv(CFG.SAMPLE_SUB)

# Set row_id as index for alignment
submission = submission.set_index('row_id')
sample_sub_indexed = sample_sub.set_index('row_id')

# Reindex: keeps only expected rows/cols, fills gaps with 0.0
submission = submission.reindex(
    index=sample_sub_indexed.index,
    columns=sample_sub_indexed.columns,
    fill_value=0.0
)

submission = submission.reset_index()  # brings row_id back as a column

# Verify shape matches sample submission exactly
assert submission.shape == sample_sub.shape, \
    f"Shape mismatch! Got {submission.shape}, expected {sample_sub.shape}"

# ⚠️ Must be exactly this filename for Kaggle to accept it
out_path = '/kaggle/working/submission.csv'
submission.to_csv(out_path, index=False)  # index=False is critical

print(f"Submission saved to {out_path}")
print(f"Shape: {submission.shape}")
print(submission.head(3))

# Confirm the file actually exists on disk
import os
assert os.path.exists(out_path), "File was not written!"
print("✓ File confirmed on disk")

Submission saved to /kaggle/working/submission.csv
Shape: (3, 235)
                                    row_id  1161364  116570  1176823  1491113  \
0   BC2026_Test_0001_S05_20250227_010002_5      0.0     0.0      0.0      0.0   
1  BC2026_Test_0001_S05_20250227_010002_10      0.0     0.0      0.0      0.0   
2  BC2026_Test_0001_S05_20250227_010002_15      0.0     0.0      0.0      0.0   

   1595929  209233  22930  22956  22961  ...  whnjay1  whtdov  whwpic1  \
0      0.0     0.0    0.0    0.0    0.0  ...      0.0     0.0      0.0   
1      0.0     0.0    0.0    0.0    0.0  ...      0.0     0.0      0.0   
2      0.0     0.0    0.0    0.0    0.0  ...      0.0     0.0      0.0   

   y00678  yebcar  yebela1  yecmac  yecpar  yehcar1  yeofly1  
0     0.0     0.0      0.0     0.0     0.0      0.0      0.0  
1     0.0     0.0      0.0     0.0     0.0      0.0      0.0  
2     0.0     0.0      0.0     0.0     0.0      0.0      0.0  

[3 rows x 235 columns]
✓ File confirmed on disk


---
## What to try next

Once you have a score on the leaderboard, here are the highest-value improvements (roughly in order):

1. **Use secondary labels** — `train_metadata.csv` has a `secondary_labels` column with additional species present in each clip. Creating soft multi-hot targets (e.g. primary=1.0, secondary=0.5) can meaningfully improve recall on rare species.

2. **Augment audio** — before embedding, add: time shift, gaussian noise, or mixup of two clips. Since embedding is slow, do augmentation online during embedding rather than storing extra files.

3. **Train a deeper head** — try 3 layers or increase hidden size to 1024. Add more `Dropout` to reduce overfitting on rare classes.

4. **Better validation split** — use `GroupKFold` on the recording filename so that clips from the same recording never span both train and val folds (avoids data leakage).

5. **Post-processing** — after getting probabilities, try multiplying by a prior (how often each species appears in training) to re-calibrate predictions toward common species.

6. **Tune the classification threshold** — the competition uses ROC-AUC, which is threshold-free, but understanding your model's precision/recall trade-off helps diagnose problems.

Good luck! 🐦